# Model Trainer - Colab GPU

In [ ]:
!pip install kagglehub matplotlib onnx onnxruntime pandas scikit-learn seaborn tensorflow tf2onnx -q

In [ ]:
import tensorflow as tf
print("GPU:", tf.config.list_physical_devices("GPU"))

## Run training

In [ ]:
# === src/config.py ===
from __future__ import annotations

In [ ]:
import logging
import os
from dataclasses import dataclass, field
from pathlib import Path

In [ ]:
logger = logging.getLogger(__name__)

In [ ]:
@dataclass(frozen=True)
class DataConfig:
    stress_condition: str = "TSST"

In [ ]:
@dataclass(frozen=True)
class ModelConfig:
    lstm_units: int = 64
    dropout_rate: float = 0.2
    learning_rate: float = 5e-4
    l2_reg: float = 1e-4
    lstm_dropout: float = 0.2
    lstm_recurrent_dropout: float = 0.0

In [ ]:
@dataclass(frozen=True)
class TrainingConfig:
    sequence_lengths: tuple[int, ...] = (120, 240, 360)
    epochs: int = 30
    batch_size: int = 64
    export_onnx: bool = True

    patience: int = 8
    reduce_lr_patience: int = 2
    reduce_lr_factor: float = 0.5
    min_lr: float = 1e-6
    positive_class_weight: float = 2.5
    decision_threshold_beta: float = 2.0

    models_dir: Path = Path(os.environ.get("MODELS_DIR_OVERRIDE", "models"))

    def model_dir(self, seq_len: int) -> Path:
        """Return (and create) the output directory for a given sequence length."""
        d = self.models_dir / f"seq_{seq_len}"
        d.mkdir(parents=True, exist_ok=True)
        return d

In [ ]:
DATA = DataConfig()
MODEL = ModelConfig()
TRAINING = TrainingConfig()

In [ ]:
# === src/data/download.py ===
from os import path

In [ ]:
import kagglehub

In [ ]:
def dataset_fetching():
    initial_path = kagglehub.dataset_download("qiriro/stress")
    return path.join(initial_path, "dataset")

In [ ]:
# === src/data/features.py ===
import numpy as np
import pandas as pd

In [ ]:
def filter_hardware_features(
    features: np.ndarray, feature_names: list[str], target_hardware: str = "MAX30102"
):
    if target_hardware == "MAX30102":
        allowed_keywords = ["RR", "RMSSD", "SDSD", "pNN", "SD1", "SD2", "HR"]
    else:
        allowed_keywords = ["RR", "RMSSD", "SDSD", "pNN", "SD1", "SD2", "HR"]

    indices = []
    filtered_names = []

    for i, name in enumerate(feature_names):
        if any(key in name.upper() for key in allowed_keywords):
            indices.append(i)
            filtered_names.append(name)

    return features[:, indices], filtered_names

In [ ]:
def extract_features(values: np.ndarray):
    v = values.flatten().astype(np.float32, copy=False)

    f1 = v
    diffs = np.diff(v, prepend=v[0])
    f3 = np.sqrt(
        pd.Series(diffs**2).rolling(window=10, min_periods=1).mean().values
    )
    f4 = pd.Series(v).rolling(window=10, min_periods=1).std().fillna(0).values
    f5 = 1.0 / (v + 1e-8)

    all_features = np.stack([f1, diffs, f3, f4, f5], axis=1).astype(np.float32, copy=False)
    all_names = ["RRI", "RRI_Diff", "RMSSD_Local", "SDNN_Local", "HR_Proxy"]

    return all_features, all_names

In [ ]:
def create_sequences(
    features: np.ndarray, labels: np.ndarray, seq_len: int
):
    X, y = [], []
    for i in range(0, len(features) - 2 * seq_len + 1):
        X.append(features[i : i + seq_len])
        next_mean = float(np.mean(labels[i + seq_len : i + 2 * seq_len]))
        y.append(1 if next_mean > 0.5 else 0)
    return np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.uint8)

In [ ]:
# === src/models/lstm.py ===
import tensorflow as tf
from tensorflow.keras import Model, layers, regularizers

In [ ]:
def build_rri_lstm(seq_len: int, n_features: int) -> Model:
    inputs = layers.Input(shape=(seq_len, n_features), name="input")

    x = layers.Bidirectional(
        layers.LSTM(
            MODEL.lstm_units,
            return_sequences=True,
            dropout=MODEL.lstm_dropout,
            recurrent_dropout=MODEL.lstm_recurrent_dropout,
            kernel_regularizer=regularizers.l2(MODEL.l2_reg),
        )
    )(inputs)
    x = layers.Bidirectional(
        layers.LSTM(
            MODEL.lstm_units // 2,
            return_sequences=True,
            dropout=MODEL.lstm_dropout,
            recurrent_dropout=MODEL.lstm_recurrent_dropout,
            kernel_regularizer=regularizers.l2(MODEL.l2_reg),
        )
    )(x)

    x = layers.GlobalAveragePooling1D()(x)

    x = layers.BatchNormalization()(x)
    x = layers.Dropout(MODEL.dropout_rate)(x)
    x = layers.Dense(
        16, activation="relu", kernel_regularizer=regularizers.l2(MODEL.l2_reg)
    )(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(MODEL.dropout_rate)(x)

    outputs = layers.Dense(1, activation="sigmoid", name="stress_prob")(x)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=MODEL.learning_rate),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.Precision(name="precision"),
        ],
    )
    return model

In [ ]:
# === src/data/load.py ===
import os
from collections.abc import Generator

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def _parse_quest(filepath: str):
    df = pd.read_csv(filepath, header=None)
    order_str = str(df.iloc[1, 0]).removeprefix("# ORDER;")
    start_str = str(df.iloc[2, 0]).removeprefix("# START;")
    end_str = str(df.iloc[3, 0]).removeprefix("# END;")
    conditions = order_str.split(";")
    starts = [float(v) for v in start_str.split(";") if v]
    ends = [float(v) for v in end_str.split(";") if v]
    return list(zip(conditions[: len(starts)], starts, ends))

In [ ]:
def _label_full_timeline(
    times_min: np.ndarray, schedule: list[tuple[str, float, float]]
) -> np.ndarray:
    if not schedule:
        return np.zeros(times_min.shape, dtype=np.intp)

    labels = np.zeros(len(times_min), dtype=np.intp)
    for condition, start, end in schedule:
        if condition == DATA.stress_condition:
            mask = (times_min >= start) & (times_min <= end)
            labels[mask] = 1
    return labels

In [ ]:
def _label_times(
    times_min: np.ndarray, schedule: list[tuple[str, float, float]]
) -> np.ndarray:
    if not schedule:
        return np.full(times_min.shape, -1, dtype=np.intp)

    schedule = sorted(schedule, key=lambda x: x[1])
    starts = np.asarray([start for _, start, _ in schedule], dtype=np.float64)
    ends = np.asarray([end for _, _, end in schedule], dtype=np.float64)
    stress_flags = np.asarray(
        [cond == DATA.stress_condition for cond, _, _ in schedule], dtype=np.intp
    )

    interval_idx = np.searchsorted(starts, times_min, side="right") - 1
    valid = (interval_idx >= 0) & (times_min <= ends[np.clip(interval_idx, 0, None)])

    labels = np.full(times_min.shape, -1, dtype=np.intp)
    labels[valid] = stress_flags[interval_idx[valid]]
    return labels

In [ ]:
def load_subject(subj_id: int, ds_path: str):
    rri_path = os.path.join(ds_path, "0. interim", "wesad", "rri", f"S{subj_id}.txt")
    quest_path = os.path.join(
        ds_path, "0. interim", "wesad", "Labels", f"S{subj_id}_quest.csv"
    )

    rri = np.loadtxt(rri_path)
    schedule = _parse_quest(quest_path)
    times_s = rri[:, 0]
    times_min = times_s / 60.0
    values = rri[:, 1]
    labels = _label_times(times_min, schedule)
    valid = labels >= 0

    v = values[valid]
    l = labels[valid].astype(np.intp)

    features, feature_names = extract_features(v)

    if len(features) > 0:
        features = (features - np.mean(features, axis=0)) / (
            np.std(features, axis=0) + 1e-8
        )

    return features, l, feature_names

In [ ]:
def load_subject_hrv(subj_id: int, ds_path: str):
    hrv_path = os.path.join(
        ds_path, "1. processed", "hrv", "wesad", "raw", "chest", f"S{subj_id}.xlsx"
    )
    quest_path = os.path.join(
        ds_path, "0. interim", "wesad", "Labels", f"S{subj_id}_quest.csv"
    )

    df = pd.read_excel(hrv_path)
    times_min = df["Time"].values
    features = df.drop(columns=["Time"]).values.astype(np.float32)
    feature_names = [c for c in df.columns if c != "Time"]

    schedule = _parse_quest(quest_path)
    labels = _label_full_timeline(times_min, schedule)
    if len(features) > 0:
        features = (features - np.mean(features, axis=0)) / (
            np.std(features, axis=0) + 1e-8
        )

    return features, labels, feature_names

In [ ]:
def load_all_subjects_hrv() -> Generator[tuple[int, np.ndarray, np.ndarray, list[str]]]:
    ds_path = dataset_fetching()
    hrv_dir = os.path.join(ds_path, "1. processed", "hrv", "wesad", "raw", "chest")
    available = sorted(
        int(f.removeprefix("S").removesuffix(".xlsx")) for f in os.listdir(hrv_dir)
    )
    for subj_id in available:
        yield subj_id, *load_subject_hrv(subj_id, ds_path)

In [ ]:
def load_all_subjects() -> Generator[tuple[int, np.ndarray, np.ndarray, list[str]]]:
    ds_path = dataset_fetching()
    rri_dir = os.path.join(ds_path, "0. interim", "wesad", "rri")
    available = sorted(
        int(f.removeprefix("S").removesuffix(".txt")) for f in os.listdir(rri_dir)
    )
    for subj_id in available:
        yield subj_id, *load_subject(subj_id, ds_path)

In [ ]:
# === src/training/export.py ===
from pathlib import Path

In [ ]:
import tensorflow as tf
from tensorflow.keras import Model

In [ ]:
def export_to_onnx(model: Model, seq_len: int, n_features: int, model_dir: Path):
    import tf2onnx

    onnx_path = model_dir / "model.onnx"
    input_name = "input"

    if hasattr(model, "input_names") and model.input_names:
        input_name = model.input_names[0]

    spec = (
        tf.TensorSpec((None, seq_len, n_features), tf.float32, name=input_name),
    )

    try:
        model_proto, _ = tf2onnx.convert.from_keras(
            model,
            input_signature=spec,
            opset=13,
            output_path=str(onnx_path),
        )
        print(f"Successfully converted to ONNX: {onnx_path}")
    except Exception as e:
        print(
            f"Direct ONNX conversion failed: {e}. Trying via temporary SavedModel..."
        )
        import subprocess
        import sys
        import tempfile

        with tempfile.TemporaryDirectory() as temp_dir:
            temp_saved_model = str(Path(temp_dir) / "temp_save")
            tf.saved_model.save(model, temp_saved_model)

            cmd = [
                sys.executable,
                "-m",
                "tf2onnx.convert",
                "--saved-model",
                temp_saved_model,
                "--output",
                str(onnx_path),
                "--opset",
                "13",
            ]
            result = subprocess.run(cmd, capture_output=True, text=True)
            if result.returncode == 0:
                print(
                    f"Successfully converted to ONNX via temporary SavedModel: {onnx_path}"
                )
            else:
                print(f"ONNX conversion CLI fallback failed: {result.stderr}")

In [ ]:
# === src/training/trainer.py ===
from __future__ import annotations

In [ ]:
import logging
from pathlib import Path

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from tensorflow.keras import Model

In [ ]:
logger = logging.getLogger(__name__)

In [ ]:
def _validate_inputs(
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    seq_len: int,
) -> None:
    if X_train.ndim != 3:
        raise ValueError(
            f"X_train must be 3-D (samples, timesteps, features), got {X_train.shape}"
        )
    if X_train.shape[1] != seq_len:
        raise ValueError(f"X_train timesteps ({X_train.shape[1]}) != seq_len ({seq_len})")
    if X_train.shape[0] != y_train.shape[0]:
        raise ValueError(
            f"X_train / y_train sample count mismatch: {X_train.shape[0]} vs {y_train.shape[0]}"
        )
    if X_val.shape[2] != X_train.shape[2]:
        raise ValueError(
            f"Feature count mismatch: train={X_train.shape[2]}, val={X_val.shape[2]}"
        )

In [ ]:
def _compute_positive_rate(y: np.ndarray) -> float:
    if len(y) == 0:
        raise ValueError("Cannot compute a positive rate for an empty label vector.")
    return float(np.mean(y))

In [ ]:
def _build_loss(y_train: np.ndarray) -> tf.keras.losses.Loss:
    positive_rate = float(np.clip(_compute_positive_rate(y_train), 0.05, 0.5))
    alpha = float(np.clip((1.0 - positive_rate) * TRAINING.positive_class_weight, 0.5, 0.9))
    logger.info("Using focal loss with alpha=%.3f, gamma=2.0", alpha)
    return tf.keras.losses.BinaryFocalCrossentropy(
        gamma=2.0,
        alpha=alpha,
        from_logits=False,
        apply_class_balancing=False,
    )

In [ ]:
def _build_callbacks() -> list[tf.keras.callbacks.Callback]:
    return [
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=TRAINING.patience,
            restore_best_weights=True,
            verbose=1,
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=TRAINING.reduce_lr_factor,
            patience=TRAINING.reduce_lr_patience,
            min_lr=TRAINING.min_lr,
            verbose=1,
        ),
    ]

In [ ]:
def _find_best_threshold(
    y_true: np.ndarray,
    y_prob: np.ndarray,
) -> tuple[float, dict[str, float]]:
    from sklearn.metrics import fbeta_score, precision_score, recall_score

    best_threshold = 0.5
    best_score = -1.0
    best_metrics: dict[str, float] = {}

    for threshold in np.linspace(0.1, 0.9, 81):
        y_pred = (y_prob >= threshold).astype(np.uint8)
        score = fbeta_score(
            y_true,
            y_pred,
            beta=TRAINING.decision_threshold_beta,
            zero_division=0,
        )
        if score > best_score:
            best_score = float(score)
            best_threshold = float(threshold)
            best_metrics = {
                "precision": float(precision_score(y_true, y_pred, zero_division=0)),
                "recall": float(recall_score(y_true, y_pred, zero_division=0)),
                "fbeta": float(score),
            }

    return best_threshold, best_metrics

In [ ]:
def _save_training_curves(
    history: tf.keras.callbacks.History,
    model_dir: Path,
) -> None:
    try:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

        ax1.plot(history.history["loss"], label="train")
        ax1.plot(history.history["val_loss"], label="val")
        ax1.set_title("Loss")
        ax1.set_xlabel("Epoch")
        ax1.legend()

        ax2.plot(history.history["accuracy"], label="train")
        ax2.plot(history.history["val_accuracy"], label="val")
        ax2.set_title("Accuracy")
        ax2.set_xlabel("Epoch")
        ax2.legend()

        plt.tight_layout()
        out = model_dir / "training_curves.png"
        plt.savefig(out, dpi=150)
        logger.info("Saved training curves -> %s", out)
    except Exception:
        logger.exception("Failed to save training curves - continuing.")
    finally:
        plt.close("all")

In [ ]:
def _save_model(model: Model, model_dir: Path) -> None:
    out = model_dir / "model.keras"
    model.save(str(out))
    logger.info("Saved Keras model -> %s", out)

In [ ]:
def train_model(
    seq_len: int,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
) -> tuple[Model, tf.keras.callbacks.History]:

    _validate_inputs(X_train, y_train, X_val, y_val, seq_len)

    n_features = X_train.shape[2]
    model_dir = TRAINING.model_dir(seq_len)

    logger.info(
        "Training seq_len=%d | samples=%d | features=%d | dir=%s",
        seq_len,
        len(X_train),
        n_features,
        model_dir,
    )

    model = build_rri_lstm(seq_len, n_features)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=MODEL.learning_rate),
        loss=_build_loss(y_train),
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.AUC(curve="PR", name="pr_auc"),
            tf.keras.metrics.Recall(name="recall"),
            tf.keras.metrics.Precision(name="precision"),
        ],
    )

    idx = np.random.permutation(len(X_train))
    X_train, y_train = X_train[idx], y_train[idx]

    train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    train_ds = train_ds.batch(TRAINING.batch_size).prefetch(tf.data.AUTOTUNE)
    val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    val_ds = val_ds.batch(TRAINING.batch_size).prefetch(tf.data.AUTOTUNE)

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=TRAINING.epochs,
        callbacks=_build_callbacks(),
        verbose=1,
    )

    _save_training_curves(history, model_dir)
    if TRAINING.export_onnx:

        export_to_onnx(model, seq_len, n_features, model_dir)
    _save_model(model, model_dir)

    stopped_at = len(history.history["loss"])
    logger.info("Finished seq_len=%d after %d epochs.", seq_len, stopped_at)

    return model, history

In [ ]:
# === src/evaluate/metrics.py ===
import json

In [ ]:
import matplotlib

In [ ]:
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    fbeta_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

In [ ]:
def _find_best_threshold(y_true: np.ndarray, y_prob: np.ndarray) -> tuple[float, dict[str, float]]:
    best_threshold = 0.5
    best_score = -1.0
    best_metrics = {"precision": 0.0, "recall": 0.0, "fbeta": 0.0}

    for threshold in np.linspace(0.1, 0.9, 81):
        y_pred = (y_prob >= threshold).astype(int)
        score = fbeta_score(
            y_true,
            y_pred,
            beta=TRAINING.decision_threshold_beta,
            zero_division=0,
        )
        if score > best_score:
            best_score = float(score)
            best_threshold = float(threshold)
            best_metrics = {
                "precision": float(precision_score(y_true, y_pred, zero_division=0)),
                "recall": float(recall_score(y_true, y_pred, zero_division=0)),
                "fbeta": float(score),
            }

    return best_threshold, best_metrics

In [ ]:
def evaluate_model(model, X_test: np.ndarray, y_test: np.ndarray, seq_len: int):
    y_prob = model.predict(X_test, verbose=0).flatten()
    threshold, threshold_metrics = _find_best_threshold(y_test, y_prob)
    y_pred = (y_prob >= threshold).astype(int)

    auc = roc_auc_score(y_test, y_prob)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)

    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()

    metrics = {
        "seq_len": seq_len,
        "threshold": float(threshold),
        "threshold_precision": float(threshold_metrics["precision"]),
        "threshold_recall": float(threshold_metrics["recall"]),
        "threshold_fbeta": float(threshold_metrics["fbeta"]),
        "mean_stress_probability": float(np.mean(y_prob)),
        "max_stress_probability": float(np.max(y_prob)),
        "min_stress_probability": float(np.min(y_prob)),
        "auc": float(auc),
        "accuracy": float(acc),
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "tp": int(tp),
        "fp": int(fp),
        "tn": int(tn),
        "fn": int(fn),
    }

    print(f"\nROC-AUC Score: {auc:.4f}")
    print(f"Decision Threshold: {threshold:.2f}")
    print(f"Threshold F{TRAINING.decision_threshold_beta:.1f}: {threshold_metrics['fbeta']:.4f}")
    print(f"Accuracy: {acc:.4f}, F1: {f1:.4f}")
    print(
        "Stress probability summary - "
        f"mean: {metrics['mean_stress_probability']:.4f}, "
        f"min: {metrics['min_stress_probability']:.4f}, "
        f"max: {metrics['max_stress_probability']:.4f}"
    )
    print(f"Sample stress probabilities: {np.array2string(y_prob[:10], precision=4)}")

    model_dir = TRAINING.models_dir / f"seq_{seq_len}"
    with open(model_dir / "results.json", "w") as f:
        json.dump(metrics, f, indent=4)

    report = classification_report(
        y_test,
        y_pred,
        target_names=["not-stress", "stress"],
        zero_division=0,
    )

    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["not-stress", "stress"],
        yticklabels=["not-stress", "stress"],
    )
    plt.title(f"Confusion Matrix (seq_len={seq_len})")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")

    plot_path = model_dir / "confusion_matrix.png"
    plt.savefig(plot_path)
    plt.close()

    return report

In [ ]:
def plot_correlations(features: np.ndarray, feature_names: list[str]):
    TRAINING.models_dir.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame(features, columns=feature_names)
    plt.figure(figsize=(10, 8))
    sns.heatmap(df.corr(), annot=True, cmap="coolwarm", fmt=".2f")
    plt.title("Feature Correlation Map")
    plt.savefig(TRAINING.models_dir / "feature_correlation.png")
    plt.close()

In [ ]:
def plot_performance_summary(results: list[dict[str, float | int]]) -> None:
    if not results:
        return

    model_dir = TRAINING.models_dir
    model_dir.mkdir(parents=True, exist_ok=True)

    seq_lens = [int(item["seq_len"]) for item in results]
    x = np.arange(len(seq_lens))
    width = 0.18

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - 1.5 * width, [float(item["accuracy"]) for item in results], width, label="Accuracy")
    ax.bar(x - 0.5 * width, [float(item["precision"]) for item in results], width, label="Precision")
    ax.bar(x + 0.5 * width, [float(item["recall"]) for item in results], width, label="Recall")
    ax.bar(x + 1.5 * width, [float(item["f1"]) for item in results], width, label="F1")

    ax.set_xticks(x)
    ax.set_xticklabels([f"seq_{seq}" for seq in seq_lens])
    ax.set_ylim(0.0, 1.05)
    ax.set_ylabel("Score")
    ax.set_title("Model Performance by Sequence Length")
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.savefig(model_dir / "performance_summary.png")
    plt.close()

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - 1.5 * width, [int(item["tn"]) for item in results], width, label="TN")
    ax.bar(x - 0.5 * width, [int(item["fp"]) for item in results], width, label="FP")
    ax.bar(x + 0.5 * width, [int(item["fn"]) for item in results], width, label="FN")
    ax.bar(x + 1.5 * width, [int(item["tp"]) for item in results], width, label="TP")

    ax.set_xticks(x)
    ax.set_xticklabels([f"seq_{seq}" for seq in seq_lens])
    ax.set_ylabel("Window Count")
    ax.set_title("Confusion Matrix Counts by Sequence Length")
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.savefig(model_dir / "confusion_counts_summary.png")
    plt.close()

In [ ]:
def plot_modal_timing(run_log: list[dict[str, float | int]]) -> None:
    if not run_log:
        return

    model_dir = TRAINING.models_dir
    model_dir.mkdir(parents=True, exist_ok=True)

    seq_lens = [int(item["seq_len"]) for item in run_log]
    durations = [float(item["duration_seconds"]) for item in run_log]
    wait_times = [float(item["wait_seconds"]) for item in run_log]

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar([f"seq_{seq}" for seq in seq_lens], durations, label="Training duration")
    ax.plot([f"seq_{seq}" for seq in seq_lens], wait_times, marker="o", label="Queue-to-finish wait")
    ax.set_ylabel("Seconds")
    ax.set_title("Modal Run Timing by Sequence Length")
    ax.legend()
    ax.grid(axis="y", alpha=0.2)
    plt.tight_layout()
    plt.savefig(model_dir / "modal_timing_summary.png")
    plt.close()

In [ ]:
# === main.py ===
import os
import sys
from collections.abc import Iterable

In [ ]:
import numpy as np

In [ ]:
os.environ.setdefault("PYTHONIOENCODING", "utf-8")
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8")

In [ ]:
def run(data: dict | None = None) -> None:
    print("Data received:", data)
    print("Loading raw chest HRV data for all subjects...")
    subjects = list(load_all_subjects())

    print(f"Loaded {len(subjects)} subjects")

    filtered_subjects = []
    for subj_id, features, labels, feature_names in subjects:
        features_f, names_f = filter_hardware_features(features, feature_names)
        filtered_subjects.append((subj_id, features_f, labels, names_f))
    subjects = filtered_subjects

    if len(subjects) > 0:
        feature_names = subjects[0][3]
        n_features = len(feature_names)
        print(f"MAX30102 features ({n_features}): {feature_names}")
        plot_correlations(subjects[0][1], feature_names)
    else:
        n_features = 0

    train_subjects = [s for s in subjects if s[0] <= 13]
    val_subjects = [s for s in subjects if s[0] in (14, 15)]
    test_subjects = [s for s in subjects if s[0] >= 16]

    for seq_len in TRAINING.sequence_lengths:
        X_train, y_train = _prepare_data(train_subjects, seq_len)
        X_val, y_val = _prepare_data(val_subjects, seq_len)
        X_test, y_test = _prepare_data(test_subjects, seq_len)
        _train_local_wrapper(seq_len, n_features, X_train, y_train, X_val, y_val, X_test, y_test)

    print(f"\nAll models saved to: {TRAINING.models_dir}")
    print(
        "\nArchitecture: [RRI sensor readings] -> sequence -> LSTM -> stress prediction"
    )

In [ ]:
def _prepare_data(
    subj_list: Iterable[tuple[int, np.ndarray, np.ndarray, list[str]]],
    seq_len: int,
) -> tuple[np.ndarray, np.ndarray]:
    subj_items = list(subj_list)
    all_X, all_y = [], []
    for _, v, l, _ in subj_items:
        X_sub, y_sub = create_sequences(v, l, seq_len)
        if len(X_sub) > 0:
            all_X.append(X_sub)
            all_y.append(y_sub)

    if not all_X or not all_y:
        raise ValueError(f"No training sequences could be created for seq_len={seq_len}")

    X = np.concatenate(all_X).astype(np.float32, copy=False)
    y = np.concatenate(all_y).astype(np.uint8, copy=False)
    return X, y

In [ ]:
def _train_local_wrapper(
    seq_len: int,
    n_features: int,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
) -> None:
    print(f"\n{'=' * 60}")
    print(f"Sensor sequence -> LSTM (seq_len={seq_len})")
    print(f"Input: ({seq_len}, {n_features}) HRV features -> LSTM -> stress probability")
    print(f"{'=' * 60}")
    print(f"Train sequences: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
    print(f"Stress ratio - Train: {y_train.mean():.3f}, Val: {y_val.mean():.3f}, Test: {y_test.mean():.3f}")

    _train_local(seq_len, X_train, y_train, X_val, y_val, X_test, y_test)

In [ ]:
def _train_local(
    seq_len: int,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_val: np.ndarray,
    y_val: np.ndarray,
    X_test: np.ndarray,
    y_test: np.ndarray,
):

    model, _history = train_model(seq_len, X_train, y_train, X_val, y_val)
    report = evaluate_model(model, X_test, y_test, seq_len)
    print(f"\nClassification Report (seq_len={seq_len}):\n{report}")

In [ ]:
if __name__ == "__main__":
    run()

## Save models to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, shutil, glob
out = "/content/drive/MyDrive/colab/model-trainer/models"
if os.path.exists(out):
    shutil.rmtree(out)
for src in glob.glob("models/wesad_lstm/**", recursive=True):
    if os.path.isfile(src):
        dst = src.replace("models/wesad_lstm", out)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy2(src, dst)
print(f"Saved to {out}")
